Premium PV and Wind Power sites use measured power to train the site model. Measurements are managed with the [`pv_power_site_measurements`](https://solcast.github.io/solcast-api-python-sdk/pv_power_site_measurements/) and [`wind_power_site_measurements`](https://solcast.github.io/solcast-api-python-sdk/wind_power_site_measurements/) modules.

In this example, we will use the SDK for the following.

* Upload measurements in batches of up to 1,000 records
* Download all measurements for a period with `skip` and `take`

The upload example writes data to the site. Replace `your-premium-pv-site` with the `resource_id` of your own Premium PV Power site.

In [1]:
import pandas as pd

from solcast import pv_power_site_measurements

## Upload Measurements in Batches

Each create request accepts up to 1,000 measurements. Split a larger list into batches of 1,000 and send one request for each batch.

In [ ]:
resource_id = "cbf6-a872-1cb2-566f"
measurements = [
    {"period_end": "2026-09-28T00:30:00Z", "period": "PT30M", "power": 1.25},
    {"period_end": "2026-09-28T01:00:00Z", "period": "PT30M", "power": 1.4},
]

# NOTE: this writes measurements to the site!
for i in range(0, len(measurements), 1000):
    res = pv_power_site_measurements.create_pv_site_measurements(
        resource_id=resource_id,
        measurements=measurements[i : i + 1000],
        format="json",
    )
    if not res.success:
        raise RuntimeError(res.exception)
    print(res.to_dict())

Sub-unit measurements use the same batch limit. Each record must include the `sub_unit` label for its inverter.

In [ ]:
sub_unit_measurements = [
    {
        "sub_unit": "inverter-1",
        "period_end": "2026-09-28T00:30:00Z",
        "period": "PT30M",
        "power": 0.65,
    },
    {
        "sub_unit": "inverter-1",
        "period_end": "2026-09-28T01:00:00Z",
        "period": "PT30M",
        "power": 0.72,
    },
]

# NOTE: this writes sub-unit measurements to the site!
for i in range(0, len(sub_unit_measurements), 1000):
    res = pv_power_site_measurements.create_pv_sub_unit_site_measurements(
        resource_id=resource_id,
        measurements=sub_unit_measurements[i : i + 1000],
        format="json",
    )
    if not res.success:
        raise RuntimeError(res.exception)
    print(res.to_dict())

## Display Sub-Unit Measurements

Pass a `sub_unit` label to retrieve measurements for one inverter. The query uses the same `skip` and `take` pagination as site measurements.

In [ ]:
sub_unit_results = []
sub_unit_skip = 0
sub_unit_take = 100
sub_unit = "inverter-1"

while True:
    res = pv_power_site_measurements.get_pv_sub_unit_site_measurements(
        resource_id=resource_id,
        sub_unit=sub_unit,
        start="2026-09-28T00:00:00Z",
        end="2026-09-29T00:00:00Z",
        skip=sub_unit_skip,
        take=sub_unit_take,
    )
    if not res.success:
        raise RuntimeError(res.exception)

    page = res.to_dict()
    page_records = page["results"]
    sub_unit_results.extend(page_records)
    sub_unit_skip = page["offset"] + len(page_records)
    if not page_records or sub_unit_skip >= page["total"]:
        break

if sub_unit_results:
    sub_unit_df = pd.DataFrame(sub_unit_results).set_index("period_end")
    sub_unit_df.index = pd.to_datetime(sub_unit_df.index)
    display(sub_unit_df)
else:
    print(f"No measurements found for sub-unit: {sub_unit}")

Use the same pattern with `create_pv_sub_unit_site_measurements` for inverter-level data. Each sub-unit record must include a `sub_unit` label.

## Download All Pages

Measurements are returned in pages. Each response has `offset`, `total`, and `results` fields. Increase `skip` after each page until all records are collected.

In [ ]:
results = []
skip, take = 0, 100

while True:
    res = pv_power_site_measurements.get_pv_site_measurements(
        resource_id=resource_id,
        start="2026-09-28T00:00:00Z",
        end="2026-09-29T00:00:00Z",
        skip=skip,
        take=take,
    )
    page = res.to_dict()
    records = page["results"]
    results.extend(records)
    skip = page["offset"] + len(records)
    if not records or skip >= page["total"]:
        break

if results:
    df = pd.DataFrame(results).set_index("period_end")
    df.index = pd.to_datetime(df.index)
    df["power"].plot(figsize=[15, 4])
else:
    print("No PV measurements were returned for this time range.")

## Delete Measurements

Deletion removes records in the selected time range. Review the resource ID and bounds before setting `confirm_delete` to `True`. For sub-unit data, use `delete_pv_sub_unit_site_measurements` and pass the `sub_unit` label.

In [ ]:
confirm_delete = False
if confirm_delete:
    res = pv_power_site_measurements.delete_pv_site_measurements(
        resource_id=resource_id,
        start="2026-09-28T00:00:00Z",
        end="2026-09-28T02:00:00Z",
    )
    if not res.success:
        raise RuntimeError(res.exception)
    print(res.to_dict())
else:
    print("Deletion is disabled.")

## Delete Sub-Unit Measurements

Set `sub_unit` and the required time range to delete only that inverter's records. Keep the confirmation flag false until you are ready to delete. For Wind, use `delete_wind_sub_unit_site_measurements`.

In [ ]:
confirm_sub_unit_delete = True
if confirm_sub_unit_delete:
    res = pv_power_site_measurements.delete_pv_sub_unit_site_measurements(
        resource_id=resource_id,
        sub_unit=sub_unit,
        start="2026-09-28T00:00:00Z",
        end="2026-09-28T02:00:00Z",
    )
    if not res.success:
        raise RuntimeError(res.exception)
    print(res.to_dict())
else:
    print("Sub-unit deletion is disabled.")

## Wind Sites

The `wind_power_site_measurements` module uses the same workflow with a Premium Wind Power `resource_id`.

* Upload site records with `create_wind_site_measurements`.
* Retrieve site records with `get_wind_site_measurements` and `skip`/`take`.
* Upload turbine records with `create_wind_sub_unit_site_measurements`; add a `sub_unit` label to each record.
* Retrieve turbine records with `get_wind_sub_unit_site_measurements` and pass `sub_unit`, `skip`, and `take`.
* Delete site records with `delete_wind_site_measurements` and required `start`/`end` bounds.
* Delete turbine records with `delete_wind_sub_unit_site_measurements`, `sub_unit`, and required `start`/`end` bounds.

Wind measurements can also include `wind_speed_hub_height` and `wind_direction_hub_height`.